### Structured Output Streaming with Microsoft Agent Framework

**Fundamental logic:** Structured output asks the model to return data that conforms to a defined schema instead of unstructured prose.


In [1]:
# Fundamental logic: Pinned dependencies provide compatible Agent Framework, Azure SDK, and environment-loading
# behavior.

%pip install agent-framework==1.0.0b251209 python-dotenv azure-ai-projects==2.0.0b2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting Up the Environment

**Fundamental logic:** The next cell loads the connection settings required to create the Foundry client and agent.


In [2]:
# Fundamental logic: The endpoint identifies the Foundry project, while the deployment name chooses the model used
# for extraction.

import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://ajay-agent-project111-resource.services.ai.azure.com/api/projects/ajay-agent-project111
Model:  ajay-gpt-4o


### Creating the Foundry Client and Agent

**Fundamental logic:** The agent is configured as an information extractor and connected to its own conversation.


In [3]:
# Fundamental logic: This async setup creates authenticated clients, conversation state, and an agent whose
# instructions define the extraction task.

from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential
from azure.ai.projects.aio import AIProjectClient

async def create_agent():
    credential = AzureCliCredential()
    
    # creating the Foundry Project Client
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )

    # creating a conversation using the OpenAI Client
    openai_client = project_client.get_openai_client()
    conversation = await openai_client.conversations.create()
    conversation_id = conversation.id
    print("Conversation ID: ", conversation_id)
    
    # creating the Azure AI Client to interact with the Agent in Foundry
    agent_client = AzureAIClient(
        project_client = project_client,
        conversation_id = conversation_id,
        model_deployment_name=model
    )
    
    # creating an agent in Foundry
    agent = agent_client.create_agent(
        name="JobCandidateExtractor",
        instructions="You are an HR assistant that extracts structured information about a job candidate from text."
    )
    return agent, credential, agent_client

agent, credential, agent_client = await create_agent()

Conversation ID:  conv_21ed003bfcd674c700oxeFQV1jJBK1b8Bbrv0Gq8r1BavD1hdb


### Creating a Structured Output Schema using Pydantic

**Fundamental logic:** Pydantic turns the expected response shape into a validated Python model.


In [4]:
# Fundamental logic: The schema defines required fields and rejects unexpected fields, making downstream code more
# predictable.

from pydantic import BaseModel, ConfigDict

class CandidateProfile(BaseModel):
    """Structured job candidate profile"""
    name: str
    experience_years: int
    skills: list[str]
    current_role: str
    model_config = ConfigDict(extra="forbid")

### Run the Agent with Structured Output Streaming

**Fundamental logic:** Passing the schema as response_format asks the agent to produce and validate a CandidateProfile.


In [5]:
# Fundamental logic: The model extracts facts from free text, and Agent Framework parses the response into the typed
# Pydantic object.

text_input = """
    Hi, I’m Alice Johnson. I've been a software engineer for 6 years,
    mainly working with Python, Azure, and React. Currently a senior developer at Contoso Ltd.
    """

response = await agent.run(text_input, response_format=CandidateProfile)

print(response)

{"name":"Alice Johnson","experience_years":6,"skills":["Python","Azure","React"],"current_role":"Senior Developer at Contoso Ltd"}


In [6]:
# Fundamental logic: Typed fields can be consumed directly after checking that parsing produced a value.

if response.value:
    print("name:", response.value.name)
    print("experience_years:", response.value.experience_years)
    print("skills:", response.value.skills)
    print("current_role:", response.value.current_role)
else:
    print("Failed to parse response")

name: Alice Johnson
experience_years: 6
skills: ['Python', 'Azure', 'React']
current_role: Senior Developer at Contoso Ltd
